## Preprocess _C. elegans_ transcriptome data  from the [CeNGEN project](https://www.cengen.org/downloads/)

 _C. elegans_ is the only multicellular organism for which all cells and cell types are defined, as is its entire developmental lineage. 

We downloaded and unzipped the raw reads data of "Taylor SR, Santpere G, Weinreb A, Barrett A et al. _Molecular topography of an entire nervous system_. Cell 2021 Aug 5;184(16):4329-4347.e23." from their Gene Expresion Omnibus (GEO) repository [GSE 136049](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE136049). This data represents single-cell RNA-sequencing profiles from L4 _C. elegans_ larvae from strains labeling subgroups of neuron types.

Specifically, the files we process are:

| Supplementary file | Size | Download | File type/resource | Python variable |
| :--- | :--- | :--- | :--- | :--- |
| GSE136049_all_cells_barcodes_column_names.csv.gz | 435.1 Kb | ([http](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136049&format=file&file=GSE136049%5Fall%5Fcells%5Fbarcodes%5Fcolumn%5Fnames%2Ecsv%2Egz)) | CSV | `BARCODES_PATH` | 
| GSE136049_all_cells_gene_annotations.csv.gz | 61.8 Kb | ([http](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136049&format=file&file=GSE136049%5Fall%5Fcells%5Fgene%5Fannotations%2Ecsv%2Egz)) | CSV | `GENES_PATH` |
| ~~GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf.gz~~ | ~~9.8 Mb~~ | ~~([http](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136049&format=file&file=GSE136049%5Fc%5Felegans%2EPRJNA13758%2EWS273%2Ecanonical%5Fgeneset%2Eextend3UTR%5Foptimized%2Egtf%2Egz))~~ | ~~GTF~~ | `GTF_PATH` |
| GSE136049_cell_type_annotation_lookup_table.csv.gz | 656.2 Kb | ([http](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136049&format=file&file=GSE136049%5Fcell%5Ftype%5Fannotation%5Flookup%5Ftable%2Ecsv%2Egz)) | CSV | `LOOKUP_PATH` |
| GSE136049_gene_by_barcode_count_matrix_all_cells.mtx.gz | 130.9 Mb | ([http](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE136049&format=file&file=GSE136049%5Fgene%5Fby%5Fbarcode%5Fcount%5Fmatrix%5Fall%5Fcells%2Emtx%2Egz)) | MTX |  `MTX_PATH` |

---

### Descriptions of the special file-types:

An **MTX** file is a plain-text file in the sparse Matrix Market format used to represent a large matrix efficiently.
- **Content:** In single-cell genomics, the MTX file records the expression level (count) of each gene for every single cell measured.
- **Format:** The file is sparse because most genes are not expressed in most cells, so it only stores data for non-zero entries. It uses a coordinate format, listing the coordinates (row and column) and value for each non-zero count.
- **Context:** For the MTX file to be meaningful, it is accompanied by two other tab-separated files in the same folder:
    - genes.tsv: A list of genes, where each gene's position in this list corresponds to its row index in the MTX file.
    - barcodes.tsv: A list of cell barcodes, where each barcode's position corresponds to its column index in the MTX file. 

A **GTF** (Gene Transfer Format) file is a tab-delimited text file that provides a structured, detailed annotation of a genome, including the location of genes, transcripts, and their features (like exons and coding sequences). 
- **Content:** A GTF file uses nine columns to define features, including the chromosome name, feature type (gene, exon, CDS), start and end positions, and attributes like gene_id and transcript_id.
- **Purpose:** The GTF file serves as a reference annotation file for aligning and quantifying sequencing reads. It is used to determine which genomic region a read originated from. 

The GTF file provides the crucial genomic context for interpreting the quantitative data in the MTX file.

---


### Summary of the preprocessing

In this notebook we process the raw reads data by doing the following:
- load the CeNGEN (GSE136049) sparse matrix (.mtx) + annotations (.gtf)
- keep neuronal cells only (and drops “Unannotated” by default)
- aggregate genes × neuron-class in two flavors:
	* raw UMI sums (good for absolute abundance)
	* CP10K + log1p means (good for embeddings / DR)
- map WormBase gene IDs → transcript IDs (from the provided GTF), since the GTF lacks gene_name
- write tidy CSVs (by default genes × neurons; can flip orientation with one flag)

---

Here is more detailed, step-by-step break down of the process:

1.	Read inputs
	- mtx is genes × cells (rows=genes, columns=cells).
	- genes.csv is a single column of WBGene IDs (no header).
	- barcodes.csv is a single column of cell barcodes (no header).
	- lookup.csv links each barcode → cell.type, tissue.type.
	- gtf (WS273) lacks gene_name, so we map each gene_id to the first transcript_id seen for that gene; if none, we keep the WBGene ID.

2.	Filter & align
	- Keep rows/columns aligned to the .mtx.
	- Filter to tissue.type == 'Neuron' and drop cell.type == 'Unannotated' (configurable).

3.	Aggregate to neuron classes
	- Build a one-hot matrix (cells × neuron_types).
	- Compute raw counts per type: counts = X_neuron @ S → (genes × types).

4.	Normalize for embeddings
	- Per-cell library normalization to CP10K, then log1p.
	- Take mean per neuron type (so each column is a transcriptomic profile).

5.	Map IDs and de-duplicate
	- Replace WBGene IDs by transcript IDs from the GTF (fallback to WBGene).
	- If multiple WBGene IDs map to the same transcript ID, we:
	- sum raw counts,
	- mean the log-norm means.

6.	Save
	- GSE136049_genes_by_neurons_counts.csv
	- GSE136049_genes_by_neurons_lognorm_means.csv
	- (set ORIENTATION = "neurons_by_genes" if you want rows=neurons, cols=genes)

---

Some practical notes:

- For downstream analysis like dimensionality reduction or embeddings, we recommended using the log-normalized means table.
Use the log-normalized means file. It’s far more comparable across neuron classes, and it’ll behave better with PCA/UMAP/TSNE.

- If for the orientation you want rows=neurons & cols=genes, set `ORIENTATION = "neurons_by_genes"`.

- To keep all cells instead of just the neurons, set `KEEP_ONLY_NEURONS = False` and/or `DROP_UNANNOTATED = False` if you want everything.

- Some genes can share a transcript label (rare but possible depending on annotation). We sum raw counts and average lognorm means in those cases.

In [ ]:
"""
CeNGEN (GSE136049) → Genes × Neuron-class tables
- Produces:
    * raw UMI sums per neuron class
    * CP10K + log1p means per neuron class
- Keeps original WBGene IDs (no mapping to transcript IDs)

Place these files in the `$REPO_ROOT/data/transcriptomics/input_files/` directory (or edit the paths below):
  - GSE136049_gene_by_barcode_count_matrix_all_cells.mtx
  - GSE136049_all_cells_gene_annotations.csv                                                (no header)
  - GSE136049_all_cells_barcodes_column_names.csv                                           (no header)
  - GSE136049_cell_type_annotation_lookup_table.csv                                         (barcode,cell.type,tissue.type)
  - GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf         (unused since it lacks gene_name)
"""

import git
import numpy as np
import pandas as pd
from scipy.io import mmread
from scipy import sparse
from pathlib import Path

def get_project_root() -> Path:
    """
    Returns the absolute path to the root of the Git repository.
    """
    return Path(git.Repo('.', search_parent_directories=True).working_tree_dir)

REPO_ROOT = get_project_root()
print(f"Repository root: {REPO_ROOT}")


# -----------------------------
# Config — change paths if needed
# -----------------------------
input_dir = f"{REPO_ROOT}/data/transcriptomics/Single Cell Seq/GEO (GSE 136049)/input_files"
MTX_PATH      = f"{input_dir}/GSE136049_gene_by_barcode_count_matrix_all_cells.mtx"
GENES_PATH    = f"{input_dir}/GSE136049_all_cells_gene_annotations.csv"
BARCODES_PATH = f"{input_dir}/GSE136049_all_cells_barcodes_column_names.csv"
LOOKUP_PATH   = f"{input_dir}/GSE136049_cell_type_annotation_lookup_table.csv"
GTF_PATH      = f"{input_dir}/GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf" # unused

# Keep only neurons and drop unannotated?
KEEP_ONLY_NEURONS = True
DROP_UNANNOTATED  = True

# Output orientation: "genes_by_neurons" (default) or "neurons_by_genes"
ORIENTATION = "genes_by_neurons"

# Normalization target sum for CP10K
TARGET_SUM = 1e4


# -----------------------------
# Utilities
# -----------------------------
def ensure_orientation(df, orientation="genes_by_neurons"):
    return df if orientation == "genes_by_neurons" else df.T


# -----------------------------
# 1) Load the matrix & annotations
# -----------------------------
print("Reading Matrix Market file (genes × cells)...")
X = mmread(MTX_PATH).tocsr()  # shape: (n_genes, n_cells)

print("Reading gene and barcode lists (no headers)...")
genes    = pd.read_csv(GENES_PATH, header=None)[0].astype(str).values
barcodes = pd.read_csv(BARCODES_PATH, header=None)[0].astype(str).values

print("Reading barcode → cell.type / tissue.type lookup...")
lookup = pd.read_csv(LOOKUP_PATH)  # expects columns: barcode, cell.type, tissue.type

# Basic shape sanity checks
assert X.shape[0] == len(genes),   f"Gene count mismatch: {X.shape[0]} vs {len(genes)}"
assert X.shape[1] == len(barcodes),f"Cell count mismatch: {X.shape[1]} vs {len(barcodes)}"

# Reindex lookup to match the exact column order of the matrix (barcodes)
lookup = lookup.set_index("barcode").reindex(barcodes)

# -----------------------------
# 2) Filter to neurons / drop unannotated
# -----------------------------
keep_mask = np.ones(len(barcodes), dtype=bool)

if KEEP_ONLY_NEURONS:
    keep_mask &= (lookup["tissue.type"].fillna("") == "Neuron").values

if DROP_UNANNOTATED:
    keep_mask &= (lookup["cell.type"].fillna("") != "Unannotated").values

# Subset matrix columns (cells) and the aligned lookup
keep_idx = np.where(keep_mask)[0]
Xn = X[:, keep_idx]  # genes × kept_cells
lookup_n = lookup.iloc[keep_idx].copy()

print(f"Kept {Xn.shape[1]} cells out of {X.shape[1]} (neurons only = {KEEP_ONLY_NEURONS}, drop unannotated = {DROP_UNANNOTATED})")

# -----------------------------
# 3) Build one-hot of neuron classes and aggregate RAW COUNTS
# -----------------------------
# Factorize cell types into integer codes: 0..K-1
cell_types, ct_codes = np.unique(lookup_n["cell.type"].astype(str).values, return_inverse=True)

# Build indicator S (cells × neuron_types)
rows = np.arange(Xn.shape[1])              # cell indices
cols = ct_codes                             # neuron-type code per cell
data = np.ones_like(rows)
S = sparse.csr_matrix((data, (rows, cols)), shape=(Xn.shape[1], len(cell_types)))  # cells × types

# Aggregate counts to neuron types: (genes × cells) @ (cells × types) → (genes × types)
counts_mat = Xn @ S
counts_df = pd.DataFrame(
    data=np.asarray(counts_mat.todense()),
    index=genes,
    columns=cell_types
)

# -----------------------------
# 4) CP10K + log1p means per neuron class
# -----------------------------
# Per-cell library sizes (col sums) on genes×cells => sum over rows
lib_sizes = np.asarray(Xn.sum(axis=0)).ravel()  # shape: (kept_cells,)

# Avoid divide by zero: if a cell has 0 counts, set scale to 1.0 so it stays zero after scaling
safe_lib = lib_sizes.copy()
safe_lib[safe_lib == 0] = 1.0

# Scale each column to TARGET_SUM (CP10K) by right-multiplying a diagonal matrix
scale = (TARGET_SUM / safe_lib)
D = sparse.diags(scale)                      # (cells × cells)
Xn_norm = Xn @ D                             # genes × cells, CP10K scaled

# log1p transformation: ln(x + 1)
# This reduces the effect of high-expression outliers and makes the data more normally distributed
# The +1 ensures we don't take log(0) and keeps zero counts as zero
Xn_norm = Xn_norm.tocoo(copy=True)
Xn_norm.data = np.log1p(Xn_norm.data)
Xn_norm = Xn_norm.tocsr()

# Mean per neuron class: (genes × cells) @ (cells × types) / (#cells in type)
cells_per_type = np.bincount(ct_codes, minlength=len(cell_types)).astype(float)
cells_per_type[cells_per_type == 0] = 1.0   # safety
means_mat = (Xn_norm @ S)                   # genes × types
means_mat = means_mat @ sparse.diags(1.0 / cells_per_type)

means_df = pd.DataFrame(
    data=np.asarray(means_mat.todense()),
    index=genes,
    columns=cell_types
)

# -----------------------------
# 5) Orient & save
# -----------------------------
counts_out = ensure_orientation(counts_df, ORIENTATION)
means_out  = ensure_orientation(means_df,  ORIENTATION)

output_dir = f"{REPO_ROOT}/data/transcriptomics/Single Cell Seq/GEO (GSE 136049)/output_files"
suffix = "genes_by_neurons" if ORIENTATION == "genes_by_neurons" else "neurons_by_genes"
counts_path = f"{output_dir}/GSE136049_{suffix}_counts.csv"
means_path  = f"{output_dir}/GSE136049_{suffix}_lognorm_means.csv"

counts_out.to_csv(counts_path)
means_out.to_csv(means_path)

print("Done.")
print(f"  Raw counts:      {counts_path}  (shape={counts_out.shape})")
print(f"  Lognorm means:   {means_path}   (shape={means_out.shape})")

Repository root: /Users/quileesimeon/worm-learn
Reading Matrix Market file (genes × cells)...
Reading gene and barcode lists (no headers)...
Reading barcode → cell.type / tissue.type lookup...
Kept 70296 cells out of 100955 (neurons only = True, drop unannotated = True)
Done.
  Raw counts:      /Users/quileesimeon/worm-learn/data/transcriptomics/GEO (GSE 136049)/output_files/GSE136049_genes_by_neurons_counts.csv  (shape=(22469, 130))
  Lognorm means:   /Users/quileesimeon/worm-learn/data/transcriptomics/GEO (GSE 136049)/output_files/GSE136049_genes_by_neurons_lognorm_means.csv   (shape=(22469, 130))


### Comparision of **CP10K** and **TPM**

In transcriptomics, CP10K is a sequencing depth normalization method often used in single-cell RNA-seq (scRNA-seq) data, while Transcripts Per Million (TPM) is a normalization method that accounts for both sequencing depth and gene length, commonly used for bulk RNA-seq. [1, 2, 3]  
What is CP10K? CP10K, or Counts Per 10K, is a normalization method that scales the raw transcript counts in each cell to a total of 10,000. The purpose is to address technical variations caused by differences in sequencing depth (the total number of reads per cell). [1, 4, 5, 6, 7]  
How CP10K is calculatedThe formula for CP10K for a given gene in a specific cell is as follows: [1]  

Application of CP10K 

• scRNA-seq analysis: CP10K is a common default normalization used by popular scRNA-seq analysis toolkits like Seurat and Scanpy. 
• Addressing sequencing depth differences: It corrects for the technical artifact that deeper-sequenced cells will naturally have higher raw counts. 
• Limitations: CP10K assumes that the total amount of mRNA (transcriptome size) is constant across all cells. This assumption can be problematic for cell types with naturally different transcriptome sizes, potentially leading to inaccurate results when comparing these cell types. [4, 8, 9, 10, 11]  

What is TPM? TPM, or Transcripts Per Million, is a normalization method that adjusts for both sequencing depth and the length of the gene. A longer gene will naturally produce more fragments and, thus, more reads than a shorter one at the same level of expression. TPM corrects for this bias. [2, 3, 12, 13, 14]  
How TPM is calculatedThe calculation for TPM involves two main steps: 

1. Calculate Reads Per Kilobase (RPK): Divide the raw read count for a gene by the gene's length in kilobases. 
2. Scale to per million: Sum all the RPK values for a sample and divide each RPK value by this sum, then multiply by 1,000,000. [15, 16]  

Application of TPM 

• Bulk RNA-seq analysis: TPM is a standard metric for quantifying gene expression in bulk RNA-seq data. 
• Comparing between samples: Because the total sum of all TPMs in each sample is the same, TPM values can be used to compare the relative proportion of a gene's expression across different samples. [3, 17, 18, 19]  

Key differences: CP10K vs. TPM The fundamental distinction lies in whether gene length is considered during normalization. [20]  

| Feature [1, 2, 3, 8, 21] | CP10K | TPM  |
| --- | --- | --- |
| Normalization factors | Sequencing depth (library size) only. | Sequencing depth and gene length.  |
| Calculation basis | Adjusts read counts to a fixed total per cell (e.g., 10,000). | Adjusts for gene length first, then scales to a fixed total of one million.  |
| Appropriate for | Single-cell RNA-seq, especially when using Unique Molecular Identifiers (UMIs), which are less affected by gene length bias. | Bulk RNA-seq, where gene length significantly affects read count.  |
| Best for comparing | Expression of the same gene across different cells or samples. | Relative expression of different genes within the same sample, as well as the expression of the same gene across different samples.  |
| Output value sum | The sum of normalized counts for all genes in a single cell equals 10,000. | The sum of all TPMs in a given sample is always 1,000,000.  |
